# 04 · Cost, Speed and Quality

**Module 07 · Production & Observability**

One trace tells you about *one* request. To run a desk you need to know how it is doing **overall**: what it costs, how fast it is, how often the guardrails fire, whether customers are happy. These are **metrics**, numbers computed from many traces.

We analyse real logs: `sample_logs/` holds **30 requests** from two runs of `simulate_traffic.py` (a morning of order questions, refunds, attacks and an angry customer). Reading logs costs nothing, because no model is called.

In [1]:
import json
import metrics
import observe
import quality

LOGS = "sample_logs"

traces = metrics.load_json_lines("traces.jsonl", LOGS)
report = metrics.load_report(LOGS)

print("Traces in the log:", len(traces), "(" + str(report["requests"]) + " customer requests, the rest are quality checks)")

Traces in the log: 54 (30 customer requests, the rest are quality checks)


## 1. Averages lie, percentiles do not
Say 19 customers wait 2 seconds and 1 customer waits 60 seconds. The **average** is 5 seconds, and it describes nobody. A **percentile** says: "p95 = the time that 95 out of 100 customers beat". It shows the slow tail.

*From `metrics.py`:*

```python
def percentile(values, p):
    """The value that p percent of the numbers are at or below. p95 of latencies = 'almost everyone waited less than this'.
    The average hides slow requests. Percentiles show them."""
    if len(values) == 0:
        return 0
    ordered = sorted(values)
    position = math.ceil(p / 100 * len(ordered))  # for 20 values, p95 is the 19th smallest
    if position < 1:
        position = 1
    if position > len(ordered):
        position = len(ordered)
    return ordered[position - 1]
```

In [2]:
waits = [2000] * 19 + [60000]
print("average:", sum(waits) / len(waits), "ms")
print("p50    :", metrics.percentile(waits, 50), "ms   (the typical customer)")
print("p95    :", metrics.percentile(waits, 95), "ms")
print("p100   :", metrics.percentile(waits, 100), "ms   (the unluckiest customer)")

average: 4900.0 ms
p50    : 2000 ms   (the typical customer)
p95    : 2000 ms
p100   : 60000 ms   (the unluckiest customer)


**Always watch p50 and p95.** The average hides the customers who are about to leave.

## 2. The dashboard
`metrics.build_report` reads the traces, feedback, judge grades and audit trail, and computes everything with plain loops. (The code is in `metrics.py`. It is long but simple: loop, add, count.) The text dashboard prints the result:

In [3]:
metrics.print_dashboard(report)

REQUESTS   30   outcomes: {'replied': 22, 'blocked': 6, 'human': 2}
ERRORS     0

SPEED      average 4036 ms   p50 4462 ms   p95 6387 ms   slowest 6656 ms
           average time per stage (ms):
             input_guards              1203
             router                     929
             agent:Orders agent        1662
             output_guard               839
             agent:Policy agent        1592
             agent:Refund agent        2463
             rewrite_reply              972

COST       total $0.0388   average $0.00129 per request   judge overhead $0.0352
           by stage:
             input_guards            $0.0017
             router                  $0.0009
             agent:Orders agent      $0.0005
             output_guard            $0.0332
             agent:Policy agent      $0.0008
             agent:Refund agent      $0.0017
             rewrite_reply           $0.0001
           by model:
             omni-moderation-latest  $0.0000
             

Take your time with this output. It answers most questions an owner would ask.

## 3. Where does the money go?
The `by stage` and `by model` lines say it plainly. Let us put percentages on it:

In [4]:
total = report["cost"]["total"]
print("Total cost of the 30 requests: $" + format(total, ".4f"))
print()
for stage in report["cost"]["by_stage"]:
    share = report["cost"]["by_stage"][stage] / total * 100
    print(stage.ljust(24) + "$" + format(report["cost"]["by_stage"][stage], ".4f") + "   " + format(share, ".0f") + "%")

Total cost of the 30 requests: $0.0388

input_guards            $0.0017   4%
router                  $0.0009   2%
agent:Orders agent      $0.0005   1%
output_guard            $0.0332   86%
agent:Policy agent      $0.0008   2%
agent:Refund agent      $0.0017   4%
rewrite_reply           $0.0001   0%


**One step, the output guard (the gpt-4o reviewer), costs more than everything else combined.** This is exactly the kind of fact you cannot see without tracing. It is not a bug. The reviewer is what catches replies that promise something the tools never confirmed. But now it is a **decision** you can make with numbers:

| Option | Saves | Risk |
|---|---|---|
| Review only replies that involve money (refunds) | Most of the reviewer cost | Wrong facts in other replies are not checked |
| Use a cheaper reviewer, and escalate doubtful cases to gpt-4o | A lot | Needs evals to prove it catches as much |
| Check cheap rules first (already done), then review | Already saved | None |
| Keep it as is | Nothing | None, and it costs a fraction of a cent |

Whatever you choose, **measure it with the evals from Module 05**: the cost goes down and the pass rate must not.

## 4. What would it cost at scale?
A cent here and there looks harmless. Multiplying shows the real picture. (A plain loop and the average from the dashboard.)

In [5]:
average = report["cost"]["average"]
print("Average cost per request: $" + format(average, ".5f"))
print()
print("Emails per day".ljust(16) + "Per day".rjust(10) + "Per month".rjust(12))
for emails_per_day in [100, 1000, 10000, 100000]:
    per_day = emails_per_day * average
    print(str(emails_per_day).ljust(16) + ("$" + format(per_day, ".2f")).rjust(10) + ("$" + format(per_day * 30, ".0f")).rjust(12))

Average cost per request: $0.00129

Emails per day     Per day   Per month
100                  $0.13          $4
1000                 $1.29         $39
10000               $12.90        $387
100000             $129.00       $3870


This is only the AI cost, for this traffic mix (a quarter of it was attacks, which cost almost nothing). Use your own traffic.

## 5. The cost of watching
Look at `judge overhead` in the dashboard. In this demo we graded **every** reply with gpt-4o, and the grading cost about as much as handling the customers. That is why the service grades only a **sample** (`DESK_JUDGE_SAMPLE_RATE`, one reply in four by default).

In [6]:
total = report["cost"]["total"]
overhead = report["cost"]["judge_overhead"]

print("Cost of serving the customers: $" + format(total, ".4f"))
print("Cost of grading every reply  : $" + format(overhead, ".4f"))
print()
print("Share graded".ljust(16) + "Overhead".rjust(10) + "  as a share of the serving cost")
for rate in [1.0, 0.25, 0.05]:
    print((format(rate * 100, ".0f") + "%").ljust(16) + ("$" + format(overhead * rate, ".4f")).rjust(10) + "  " + format(overhead * rate / total * 100, ".0f") + "%")

Cost of serving the customers: $0.0388
Cost of grading every reply  : $0.0352

Share graded      Overhead  as a share of the serving cost
100%               $0.0352  91%
25%                $0.0088  23%
5%                 $0.0018  5%


**Observability is not free.** Sample, so that watching costs a small fraction of working.

## 6. Where does the time go?
The dashboard lists the average time per stage. The request is a **chain**: guards, then the router, then the agent, then the reviewer, one after another. Each step waits for the previous one.

In [7]:
print("p50:", report["latency_ms"]["p50"], "ms    p95:", report["latency_ms"]["p95"], "ms")
print()
for stage in report["average_ms_by_stage"]:
    print(stage.ljust(24) + str(report["average_ms_by_stage"][stage]).rjust(6) + " ms")

p50: 4462 ms    p95: 6387 ms

input_guards              1203 ms
router                     929 ms
agent:Orders agent        1662 ms
output_guard               839 ms
agent:Policy agent        1592 ms
agent:Refund agent        2463 ms
rewrite_reply              972 ms


Ideas that a trace makes *testable*, because you can see the numbers before and after:

- **Run independent steps at the same time.** The moderation check and the injection check do not depend on each other. Running them together would cut the input guards almost in half.
- **Use a faster, smaller model** for steps that do not need a strong one (the router already does).
- **Stream the reply** so the customer starts reading sooner (Module 01).
- **Skip steps that cannot matter.** A question that touches no money does not need the strongest reviewer.

Do not guess which step to speed up. Look at the trace.

## 7. Are the guardrails firing too much, or too little?
These are the **behaviour signals** of an agent system. Sudden changes in them often reveal a problem before any customer complains.

In [8]:
guard = report["guardrails"]
total_requests = report["requests"]

print("Blocked at the door  :", guard["blocked"], "of", total_requests, "(" + format(guard["blocked"] / total_requests * 100, ".0f") + "%)")
print("Handed to a human    :", guard["sent_to_human"], "of", total_requests)
print("Replies rewritten    :", guard["reply_rewrites"])
print("Output rejections    :", guard["output_rejections"])
print()
print("Why emails were blocked:")
for reason in guard["block_reasons"]:
    print("  ", guard["block_reasons"][reason], "x", reason)

Blocked at the door  : 6 of 30 (20%)
Handed to a human    : 2 of 30
Replies rewritten    : 1
Output rejections    : 1

Why emails were blocked:
   1 x Possible prompt injection: The message a
   3 x Possible prompt injection: The customer 
   2 x The message is too long (5400 characters


In this demo, 20% of the emails were blocked, because we **sent** attacks on purpose. In a real shop, a block rate like that would be an alarm: someone is probing the desk. The number itself means nothing. **The change against normal** is the signal.

## 8. Is the quality good?
Errors are easy to see. A fluent, confident, **wrong** reply raises no error at all. So we measure quality in two ways, and they tell *different* things:

1. **A judge model** grades a sample of replies from 1 to 5 against the evidence (`quality.py`).
2. **Customers** give a thumbs up or down.

*From `quality.py`:*

```python
RUBRIC = (
    "You grade a support reply that an AI agent sent to a customer. Give a score from 1 to 5.\n"
    "5 = correct, helpful, polite and short; every fact comes from the EVIDENCE.\n"
    "4 = good, with a small weakness (a little wordy, or slightly cold).\n"
    "3 = acceptable, but it misses part of what the customer asked.\n"
    "2 = poor: unhelpful, confusing, or it states something the evidence does not support.\n"
    "1 = harmful: wrong facts, a promise the shop cannot keep, or rude.\n"
    "Judge the reply against the EVIDENCE (what the shop's tools returned), not against what you think is true."
)


class Grade(BaseModel):
    reasoning: str  # first, one or two sentences
    score: int      # 1 to 5


def judge_reply(email, reply, evidence):
    """Ask the judge model to grade one reply."""
    response = client.chat.completions.parse(
        model=JUDGE_MODEL,
        messages=[
            {"role": "system", "content": RUBRIC},
            {"role": "user", "content": "CUSTOMER EMAIL:\n" + email + "\n\nEVIDENCE:\n" + evidence + "\n\nREPLY:\n" + reply},
        ],
        response_format=Grade,
    )
    grade = response.choices[0].message.parsed

    # Never trust a model to stay inside the range: keep the score between 1 and 5
    if grade.score < 1:
        grade.score = 1
    if grade.score > 5:
        grade.score = 5
    return grade
```

In [9]:
print("Judge: average", report["quality"]["average_score"], "over", report["quality"]["graded"], "graded replies")
print("Customers: thumbs up", report["feedback"]["up"], " thumbs down", report["feedback"]["down"])

Judge: average 4.73 over 22 graded replies
Customers: thumbs up 16  thumbs down 4


The customers gave four thumbs down. Did the judge agree that those replies were bad? Let us compare the two signals for each of them.

In [10]:
grades = {}
for item in metrics.load_json_lines("quality.jsonl", LOGS):
    grades[item["request_id"]] = item

for item in metrics.load_json_lines("feedback.jsonl", LOGS):
    if item["rating"] == "down":
        request = None
        for record in traces:
            if record["request_id"] == item["request_id"]:
                request = record
        grade = grades[item["request_id"]]
        print("Email :", request["attributes"]["email_redacted"][:80])
        print("Reply :", request["attributes"]["reply_redacted"][:110])
        print("Judge :", grade["score"], "/ 5 -", grade["reasoning"][:140])
        print()

Email : I bought a coffee grinder (order A1003) three months ago and it broke. Please re
Reply : I'm sorry, but your order is not eligible for a refund since it was delivered 80 days ago, and our refund poli
Judge : 5 / 5 - The AI response effectively communicates the reason why the customer cannot receive a refund due to the order being beyond the 30-day refund

Email : Can I return an item I bought 45 days ago?
Reply : You can only ask for a refund within 30 days of delivery, so a return after 45 days is not possible unless the
Judge : 5 / 5 - The reply accurately reflects the returns policy outlined in the evidence, specifying that refunds are only applicable within 30 days unless

Email : I bought a coffee grinder (order A1003) three months ago and it broke. Please re
Reply : I'm sorry, but the order is not eligible for a refund because it was delivered 80 days ago, and our refund win
Judge : 5 / 5 - The agent's response is polite, concise, and accurately explains why the refund ca

**The customers were unhappy, and the replies were correct.** The shop's policy says no refunds after 30 days, and the desk applied it politely and accurately. A thumbs down here is about the **policy**, not about the **desk**.

This is why you need both signals. A judge cannot tell you whether customers like your policy. Customers cannot tell you whether a reply invented a fact. And neither is the truth: **read a few of the flagged requests yourself, every week.**

## 9. Grade a reply yourself
The same judge that ran on the sample logs can grade any reply. Here it grades a **good** reply and a **bad** one (it promises a refund that the tools never confirmed), against the same evidence:

In [11]:
email = "I bought a coffee grinder (order A1003) three months ago and it broke. Please refund it."
evidence = "Not eligible: delivered 80 days ago, the refund window is 30 days."

good = "I'm sorry, but order A1003 was delivered 80 days ago, which is outside our 30 day refund window, so I cannot refund it."
bad = "Good news! I have refunded order A1003 and the money will be back on your card in 3 days."

for name, reply in [("good", good), ("bad", bad)]:
    grade = quality.judge_reply(email, reply, evidence)
    print(name.upper() + " reply scored", grade.score, "/ 5")
    print("   ", grade.reasoning)

GOOD reply scored 5 / 5
    The response is polite, clear, and factually accurate according to the evidence provided. It correctly informs the customer that the refund is not possible because the request is outside the 30-day window. It is also concise and directly addresses the customer's issue.


BAD reply scored 1 / 5
    The response incorrectly promises a refund for a coffee grinder 80 days after delivery, despite the refund window only allowing returns or refunds within 30 days according to the evidence. This response promises an action that cannot be fulfilled and sets incorrect expectations for the customer, making it a harmful reply.


The judge compares the reply with the **evidence**, and that is what catches the invented refund. A judge is still a model that can be wrong, so Module 05's advice stands: check it against human scores, and treat the numbers as a signal.

## Rules of thumb
- **Watch percentiles, not just averages.**
- **Know where the money and the time go**, by stage and by model. Trace first, optimise second.
- **Observability costs money too.** Sample it.
- **Behaviour signals** (block rate, human rate, rewrites) change before the customers complain.
- **Use two quality signals**, a judge and the customers, and read real examples.

Next: **how do you find out when something is wrong, and make sure it never happens again?**